In [ ]:
import numpy as np
import pandas as pd
import os
import logging
from datetime import datetime
from pyproj import Proj
from scipy.interpolate import interp1d
from scipy.interpolate import PchipInterpolator, UnivariateSpline
import math
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.ticker import MultipleLocator
from collections import defaultdict
from numpy.lib.stride_tricks import sliding_window_view


logger = logging.getLogger(__name__)

# -----------------------------------------------------------
# 전역(글로벌) Proj 객체: UTM zone 52N (EPSG:32652)
# 위경도 좌표를 UTM(동-북) 좌표로 변환하기 위해 사용
# -----------------------------------------------------------
_proj_utm52 = Proj("epsg:32652")


class DataProcessor:
    def __init__(self, window_size=200):
        self.window_size = window_size

    @staticmethod
    def load_and_preprocess_csv(
        file_path, skiprows=100, skipfooter=100, flag=False, zone=52, window_size=200
    ):
        if flag:
            df = pd.read_csv(
                file_path,
                skiprows=skiprows,
                skipfooter=skipfooter,
                na_values=["", "nan", "NaN"],
                engine="python",
            ).fillna(0)
        else:
            df = pd.read_csv(
                file_path, skiprows=skiprows, skipfooter=skipfooter, engine="python"
            )

        df.columns = [
            "Time",
            "Accelerometer x",
            "Accelerometer y",
            "Accelerometer z",
            "Gyroscope x",
            "Gyroscope y",
            "Gyroscope z",
            "Magnetometer x",
            "Magnetometer y",
            "Magnetometer z",
            "Orientation x",
            "Orientation y",
            "Orientation z",
            "Pressure",
            "Latitude",
            "Longitude",
            "Altitude",
            "Speed_GPS",
        ]

        df["Time"] = pd.to_datetime(df["Time"], format="%Y-%m-%d %H:%M:%S.%f")
        start_dt = df["Time"].iloc[0]
        df["Elapsed Time"] = (df["Time"] - start_dt).dt.total_seconds()

        df["Acc_Norm"] = np.linalg.norm(
            df[["Accelerometer x", "Accelerometer y", "Accelerometer z"]].values, axis=1
        )
        df["Gyro_Norm"] = np.linalg.norm(
            df[["Gyroscope x", "Gyroscope y", "Gyroscope z"]].values, axis=1
        )

        e, n, df = DataProcessor.llh_to_enu(df, flag, zone)
        v_10hz, dh_10Hz = DataProcessor.interpol_vAndh(e, n)
        X, Y = DataProcessor.makeXY(df, v_10hz, dh_10Hz, window_size=window_size)

        y_e = []
        y_n = []
        dx = 0.0
        dy = 0.0
        heading = 0

        stride = 5

        # plt.plot(e, n, ".-")
        # plt.axis("equal")
        # plt.grid()
        # plt.show()

        for v, h in zip(Y[:, 0], Y[:, 1]):
            heading += h * (stride / window_size)
            dx += (v * (stride / window_size)) * np.cos(heading)
            dy += (v * (stride / window_size)) * np.sin(heading)
            y_e.append(dx)
            y_n.append(dy)

        plt.plot(e, n, ".-", label="Y_true")
        plt.plot(y_e, y_n, ".-", label="Y_pred")
        plt.grid()
        plt.axis("equal")
        plt.show()

        # plt.plot(
        #     np.cumsum(np.degrees(Y[:, 1])) * (stride / window_size), ".-", label="Y_dh"
        # )
        # plt.gca().yaxis.set_major_locator(MultipleLocator(90))
        # plt.grid()
        # plt.legend()
        # plt.show()

        return df, X, Y

    @staticmethod
    def llh_to_enu(df, flag, zone=52):
        if flag:
            valid_gps_mask = (
                df["Latitude"].notna()
                & df["Longitude"].notna()
                & (df["Latitude"].astype(str).str.strip() != "")
                & (df["Longitude"].astype(str).str.strip() != "")
            )
            valid_lat = pd.to_numeric(
                df.loc[valid_gps_mask, "Latitude"], errors="coerce"
            )
            valid_lon = pd.to_numeric(
                df.loc[valid_gps_mask, "Longitude"], errors="coerce"
            )
            final_mask = valid_lat.notna() & valid_lon.notna()
            valid_lat = valid_lat[final_mask].values
            valid_lon = valid_lon[final_mask].values
            if len(valid_lat) == 0:
                raise ValueError("유효한 GPS 데이터가 없습니다.")
            proj_enu = Proj(proj="utm", zone=zone, ellps="WGS84", south=False)
            e0, n0 = proj_enu(valid_lon[0], valid_lat[0])
            e_valid, n_valid = proj_enu(valid_lon, valid_lat)
            e_valid -= e0
            n_valid -= n0
            df["E"], df["N"] = np.nan, np.nan
            df.loc[valid_gps_mask, "E"] = e_valid
            df.loc[valid_gps_mask, "N"] = n_valid
            e = df["E"][df["E"].notna()].values
            n = df["N"][df["N"].notna()].values

        else:
            df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
            df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
            proj_enu = Proj(proj="utm", zone=zone, ellps="WGS84", south=False)
            e_all, n_all = proj_enu(df["Longitude"].values, df["Latitude"].values)
            e0, n0 = proj_enu(df["Longitude"].iloc[0], df["Latitude"].iloc[0])
            df["E"] = e_all - e0
            df["N"] = n_all - n0
            M = len(df)
            n_sec = M // 50
            e, n = [], []
            for i in range(n_sec):
                idx = min(i * 50 + 25, M - 1)
                e.append(df["E"].iloc[idx])
                n.append(df["N"].iloc[idx])
            e = np.array(e)
            n = np.array(n)

        # mask = mask_speed & mask_heading
        delta_e = np.diff(e)
        delta_n = np.diff(n)
        step = np.hypot(delta_e, delta_n)           # len = L-1

        # 3) 정지 구간
        stop_mask = step < 0.05

        # 4) 최종 마스크 (모두 길이 L-1)
        mask = (~stop_mask)
        gps_idx_all = np.arange(1, len(e))
        bad_idx = gps_idx_all[~mask]

        print(f"총 GPS 샘플: {len(e)}, 이상치 GPS 샘플: {len(bad_idx)}")

        # -------------------------------------------------------
        # (4) 센서데이터 블록 drop (1Hz GPS → 50Hz 센서)
        # -------------------------------------------------------
        drop_idx = []
        for gi in bad_idx:
            start = gi * 50
            end = (gi + 1) * 50
            drop_idx.extend(range(start, min(end, len(df))))
        df = df.drop(drop_idx).reset_index(drop=True)

        if flag:
            # --- NaN 있는 경우: 유효 GPS만 모아서 다시 ENU trajectory 계산 ---
            valid_gps_mask = (
                df["Latitude"].notna()
                & df["Longitude"].notna()
                & (df["Latitude"].astype(str).str.strip() != "")
                & (df["Longitude"].astype(str).str.strip() != "")
            )
            valid_lat = pd.to_numeric(
                df.loc[valid_gps_mask, "Latitude"], errors="coerce"
            )
            valid_lon = pd.to_numeric(
                df.loc[valid_gps_mask, "Longitude"], errors="coerce"
            )
            final_mask = valid_lat.notna() & valid_lon.notna()
            valid_lat = valid_lat[final_mask].values
            valid_lon = valid_lon[final_mask].values

            if len(valid_lat) < 2:
                raise ValueError("유효 GPS가 drop 이후 2개 미만으로 남음")

            proj_enu = Proj(proj="utm", zone=zone, ellps="WGS84", south=False)
            e0, n0 = proj_enu(valid_lon[0], valid_lat[0])
            e_valid, n_valid = proj_enu(valid_lon, valid_lat)
            e, n = e_valid - e0, n_valid - n0

        else:
            # --- NaN 없는 경우: 센서 50Hz 중간 샘플 뽑기 ---
            M = len(df)
            n_sec = M // 50
            e, n = [], []
            for i in range(n_sec):
                idx = min(i * 50 + 25, M - 1)
                e.append(df["E"].iloc[idx])
                n.append(df["N"].iloc[idx])
            e, n = np.array(e), np.array(n)

        # # -------------------------------------------------------
        # # (5) 초기 heading 정렬 + 보간
        # # -------------------------------------------------------
        dx0, dy0 = e[1] - e[0], n[1] - n[0]
        theta0 = math.atan2(dy0, dx0)
        R0 = np.array(
            [
                [math.cos(-theta0), -math.sin(-theta0)],
                [math.sin(-theta0), math.cos(-theta0)],
            ]
        )
        coords = np.vstack([e - e[0], n - n[0]])
        rotated = R0 @ coords
        e_corr, n_corr = rotated[0], rotated[1]

        return e_corr, n_corr, df
    
    @staticmethod
    def interpol_vAndh(e, n):
        delta_e = np.diff(e)
        delta_n = np.diff(n)

        v_1hz = (delta_e**2 + delta_n**2) ** 0.5
        heading_1hz = np.arctan2(delta_n, delta_e)
        dh_1hz = np.diff(np.unwrap(heading_1hz))
        origin_v = v_1hz[1:]
        origin_dh = dh_1hz

        N = len(origin_v)  # 1Hz 샘플 수
        t = np.arange(N, dtype=float)  # 0..N-1
        t_new = np.linspace(0.0, N - 1, (N - 1) * 10 + 1)  # ✅ 10Hz로 0~N-1초, 총 (N-1)*10+1개
        #t_new = np.linspace(0.0, N - 1, (N - 1) * 50 + 1)  # ✅ 50Hz로 0~N-1초, 총 (N-1)*50+1개
        pv = PchipInterpolator(t, origin_v)
        pdh = PchipInterpolator(t, origin_dh)
        
        v_10Hz = pv(t_new)
        dh_10Hz = pdh(t_new)

        return v_10Hz, dh_10Hz

    @staticmethod
    def makeXY(df, v_10Hz, dh_10Hz, window_size):
        stride = 5
        sensor_cols = [
            "Accelerometer x",
            "Accelerometer y",
            "Accelerometer z",
            "Gyroscope x",
            "Gyroscope y",
            "Gyroscope z",
            "Acc_Norm",
            "Gyro_Norm",
        ]

        values = df[sensor_cols].to_numpy()   # (N, num_features)
        N, num_features = values.shape

        # -------------------------
        # 1) X 만들기 (stride에 따라 분기)
        # -------------------------
        if stride == 1:
            # sliding_window_view로 오버헤드 최소화
            # 결과 shape: (N - window_size + 1, window_size, num_features)
            X = sliding_window_view(values, window_shape=window_size, axis=0)
        else:
            # 기존 코드 그대로 (stride=5 등)
            X_list = []
            for i in range(0, N - window_size + 1, stride):
                window = values[i : i + window_size]   # (window_size, num_features)
                X_list.append(window)
            X = np.stack(X_list, axis=0)           # (num_windows, window_size, num_features)

        Y_v = []
        Y_dh = []

        offsets = [0, 10, 20, 30]  # 1초 간격 (10Hz 기준)
        #offsets = [0, 50, 100, 150]
        for i in range(len(dh_10Hz) - max(offsets)):
            # v: 1초 단위 4개를 합
            Y_v.append(
                v_10Hz[i + offsets[0]]
                + v_10Hz[i + offsets[1]]
                + v_10Hz[i + offsets[2]]
                + v_10Hz[i + offsets[3]]
            )

            Y_dh.append(
                dh_10Hz[i + offsets[0]]
                + dh_10Hz[i + offsets[1]]
                + dh_10Hz[i + offsets[2]]
                + dh_10Hz[i + offsets[3]]
            )  
        Y = np.stack([Y_v, Y_dh], axis=1)
        X = X[: len(Y)]
        return X, Y

    @staticmethod
    def load_and_preprocess_csv_test(file_path, skiprows=50):
        # ... 기존 테스트용 전처리 로직 그대로 유지 ...
        df = pd.read_csv(file_path, skiprows=skiprows, skipfooter=100, engine="python")

        df.columns = [
            "Time",
            "Accelerometer x",
            "Accelerometer y",
            "Accelerometer z",
            "Gyroscope x",
            "Gyroscope y",
            "Gyroscope z",
            "Magnetometer x",
            "Magnetometer y",
            "Magnetometer z",
            "Orientation x",
            "Orientation y",
            "Orientation z",
            "Pressure",
            "Latitude",
            "Longitude",
            "Altitude",
            "Speed_GPS",
        ]
        df["Time"] = pd.to_datetime(df["Time"], format="%Y-%m-%d %H:%M:%S.%f")
        start_dt = df["Time"].iloc[0]
        df["Elapsed Time"] = (df["Time"] - start_dt).dt.total_seconds()

        df["Acc_Norm"] = np.linalg.norm(
            df[["Accelerometer x", "Accelerometer y", "Accelerometer z"]].values, axis=1
        )
        df["Gyro_Norm"] = np.linalg.norm(
            df[["Gyroscope x", "Gyroscope y", "Gyroscope z"]].values, axis=1
        )

        return df

In [ ]:
config = {
    "looking_left01.csv": {"skiprows": 300, "flag": False, "zone": 52},
    "looking_left02.csv": {"skiprows": 150, "flag": False, "zone": 52},
    "looking_left03.csv": {"skiprows": 250, "flag": False, "zone": 52},
    "looking_left04.csv": {"skiprows": 150, "flag": False, "zone": 52},
    "looking_left05.csv": {"skiprows": 300, "flag": True, "zone": 52},
    
    "looking_right01.csv": {"skiprows": 200, "flag": False, "zone": 52},
    "looking_right02.csv": {"skiprows": 100, "flag": False, "zone": 52},
    "looking_right03.csv": {"skiprows": 200, "flag": False, "zone": 52},
    "looking_right04.csv": {"skiprows": 350, "flag": True, "zone": 52},
    
    "looking_lr01.csv": {"skiprows": 100, "flag": False, "zone": 52},
    "looking_lr02.csv": {"skiprows": 500, "flag": True, "zone": 52},
    
    "swing_left01.csv": {"skiprows": 200, "flag": False, "zone": 52},
    "swing_left02.csv": {"skiprows": 400, "flag": False, "zone": 52},
    "swing_left03.csv": {"skiprows": 300, "flag": False, "zone": 52},
    "swing_left04.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_left05.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_left06.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_left07.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_left08.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_left09.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_left10.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_left11.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_left12.csv": {"skiprows": 300, "flag": True, "zone": 52},

    "swing_right01.csv": {"skiprows": 150, "flag": False, "zone": 52},
    "swing_right02.csv": {"skiprows": 300, "flag": False, "zone": 52},
    "swing_right03.csv": {"skiprows": 300, "flag": False, "zone": 52},
    "swing_right04.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_right05.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_right06.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_right07.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_right08.csv": {"skiprows": 500, "flag": True, "zone": 52},
    "swing_right09.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_right10.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_right11.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_right12.csv": {"skiprows": 300, "flag": True, "zone": 52},
    
    "calling_left01.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    "calling_left02.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    "calling_left03.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    "calling_left04.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    "calling_left05.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    "calling_left06.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    
    "calling_right01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_right02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_right03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_right04.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_right05.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_right06.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "out_looking_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "out_looking_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "out_swing_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "out_swing_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "looking_jw_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jw_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jw_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "looking_jw_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jw_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jw_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "swing_jw_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jw_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jw_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "swing_jw_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jw_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jw_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "calling_jw_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jw_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jw_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "calling_jw_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jw_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jw_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "looking_jh_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jh_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jh_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "looking_jh_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jh_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jh_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "swing_jh_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jh_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jh_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "swing_jh_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jh_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jh_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "calling_jh_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jh_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jh_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "calling_jh_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jh_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jh_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "looking_hr_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_hr_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_hr_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_hr_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_hr_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_hr_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "swing_hr_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_hr_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_hr_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_hr_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_hr_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_hr_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "calling_hr_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_hr_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_hr_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_hr_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_hr_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_hr_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
      
}

default_config = {"skiprows": 500, "flag": False, "zone": 52}

# =========================
# 전역 설정
# =========================
BASE_DIR = os.getcwd()
FS = 50  # Hz


learn_data_paths = [
        #tester1
        #보고걷기 좌회전
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left01.csv"), # 2분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left02.csv"), # 3분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left04.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left05.csv"), # 10분 
        
        #보고걷기 우회전 
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_right01.csv"), # 3분
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_right02.csv"), # 4분
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_right03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_right04.csv"), # 10분 
        
        #스윙 좌회전
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left01.csv"), # 2.5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left04.csv"), # 10분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left05.csv"), # 5분
        
        #스윙 우회전 
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right01.csv"), # 2.5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right04.csv"), # 10분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right05.csv"), # 5분

        
        #전화받기 좌회전 30분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left01.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left02.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left03.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left04.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left05.csv"), # 5분 
    
        #전화받기 우회전 30분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right01.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right02.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right03.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right04.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right05.csv"), # 5분 

        
        # ============================================================
        # ============================================================
        # tester2
        # 보고걷기 좌회전 15분 
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_l02.csv"), # 5분
        
        # 보고걷기 우회전 15분
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_r02.csv"), # 5분
        
        # 스윙 좌회전 15분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_l02.csv"), # 5분
        
        # 스윙 우회전 15분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_r02.csv"), # 5분
        
        # 전화받기 좌회전 15분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_l02.csv"), # 5분
        
        # 전화받기 우회전 15분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_r02.csv"), # 5분
        
        # ============================================================
        # ============================================================
        # tester3 
        # 보고걷기 좌회전 15분 
        os.path.join(BASE_DIR, "data", "tester3", "looking_jh_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "looking_jh_l02.csv"), # 5분
        
        # 보고걷기 우회전 15분
        os.path.join(BASE_DIR, "data", "tester3", "looking_jh_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "looking_jh_r02.csv"), # 5분
        
        # 스윙 좌회전 15분
        os.path.join(BASE_DIR, "data", "tester3", "swing_jh_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "swing_jh_l02.csv"), # 5분
        
        # 스윙 우회전 15분
        os.path.join(BASE_DIR, "data", "tester3", "swing_jh_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "swing_jh_r02.csv"), # 5분
        
        # 전화받기 좌회전 15분
        os.path.join(BASE_DIR, "data", "tester3", "calling_jh_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "calling_jh_l02.csv"), # 5분
        
        # 전화받기 우회전 15분
        os.path.join(BASE_DIR, "data", "tester3", "calling_jh_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "calling_jh_r02.csv"), # 5분
        
        # ============================================================
        # ============================================================
        # tester4
        
        # 보고걷기 좌회전 15분 
        os.path.join(BASE_DIR, "data", "tester4", "looking_hr_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "looking_hr_l02.csv"), # 5분
        
        # 보고걷기 우회전 15분
        os.path.join(BASE_DIR, "data", "tester4", "looking_hr_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "looking_hr_r02.csv"), # 5분
        
        # 스윙 좌회전 15분
        os.path.join(BASE_DIR, "data", "tester4", "swing_hr_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "swing_hr_l02.csv"), # 5분
        
        # 스윙 우회전 15분
        os.path.join(BASE_DIR, "data", "tester4", "swing_hr_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "swing_hr_r02.csv"), # 5분 
        
        # 전화받기 좌회전 15분
        os.path.join(BASE_DIR, "data", "tester4", "calling_hr_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "calling_hr_l02.csv"), # 5분
        
        # 전화받기 우회전 15분
        os.path.join(BASE_DIR, "data", "tester4", "calling_hr_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "calling_hr_r02.csv"), # 5분
        
]

val_data_paths = [
        
        # 보고걷기, 스윙, 전화받기 좌, 우 각 5분씩
        # tester1 검증 데이터
        #os.path.join(BASE_DIR, "data", "learn_data", "looking", "looking_lr01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_lr02.csv"), # 10분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left06.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right06.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left06.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right06.csv"), # 5분
        
        # tester2 검증 데이터 
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_l03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_r03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_l03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_r03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_l03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_r03.csv"), # 5분
        
        # tester3 검증 데이터
        os.path.join(BASE_DIR, "data", "tester3", "looking_jh_l03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "looking_jh_r03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "swing_jh_l03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "swing_jh_r03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "calling_jh_l03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "calling_jh_r03.csv"), # 5분
        
        # tester4 검증 데이터 
        os.path.join(BASE_DIR, "data", "tester4", "looking_hr_l03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "looking_hr_r03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "swing_hr_l03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "swing_hr_r03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "calling_hr_l03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "calling_hr_r03.csv"), # 5분
    ]

In [ ]:
# =========================
# 그룹 로딩 & 길이 합산
# =========================
def load_group(paths, loader_fn, cfg=None, default_cfg=None, fs=FS):
    """
    한 그룹(paths) 로드/전처리, 파일별 옵션 자동 적용.
      - loader_fn(path, **opts) -> (df, X, Y)
      - cfg: 파일명별 옵션 dict
      - default_cfg: 기본 옵션 dict
    """
    cfg = cfg or {}
    default_cfg = default_cfg or {}
    df_list, X_list, Y_list = [], [], []
    total_min = 0.0

    for idx, p in enumerate(paths, start=1):
        fname = Path(p).name
        opts = {**default_cfg, **cfg.get(fname, {})}  # default + per-file override

        try:
            df_temp, X_temp, Y_temp = loader_fn(p, **opts)
        except Exception as e:
            print(f"[에러] {fname}: {e}")
            continue

        df_list.append(df_temp)
        X_list.append(X_temp)
        Y_list.append(Y_temp)

        minutes = len(df_temp) / fs / 60.0
        print(
            f"[{idx:02d}] {fname:<32} 길이:{len(df_temp):7d}  ≈ {minutes:6.2f} 분  opts={opts}"
        )
        total_min += minutes

    print(f"--> 그룹 합계: {total_min:.2f} 분\n")
    return df_list, X_list, Y_list, total_min


# =========================
# 좌/우/혼합 분량 요약
# =========================
def summarize_turn_minutes(df_list, paths, fs=FS):
    """
    df_list: 각 파일의 DataFrame 리스트
    paths  : 각 파일의 경로 리스트 (df_list와 동일 순서)
    fs     : 샘플링 주파수(Hz)
    return : (per_file_rows, totals)
    """
    rows = []
    tot_left = 0.0
    tot_right = 0.0
    for df, p in zip(df_list, paths):
        fname = Path(p).name
        name = fname.lower()
        minutes = len(df) / fs / 60.0

        if ("lr" in name) or ("left" in name and "right" in name):
            # 혼합 데이터 → 좌/우 반반
            left_min = minutes / 2.0
            right_min = minutes / 2.0
            tag = "lr(50/50)"
        elif "left" in name:
            left_min = minutes
            right_min = 0.0
            tag = "left"
        elif "right" in name:
            left_min = 0.0
            right_min = minutes
            tag = "right"
        else:
            left_min = 0.0
            right_min = 0.0
            tag = "unknown"

        tot_left += left_min
        tot_right += right_min
        rows.append(
            {
                "file": fname,
                "minutes": round(minutes, 2),
                "tag": tag,
                "left_min": round(left_min, 2),
                "right_min": round(right_min, 2),
            }
        )

    totals = {
        "left": round(tot_left, 2),
        "right": round(tot_right, 2),
        "total": round(tot_left + tot_right, 2),
    }
    return rows, totals


def pretty_print_summary(title, rows, totals):
    print(f"\n=== {title} ===")
    print(f"{'file':32} {'min':>7}  {'tag':10} {'left_min':>9} {'right_min':>10}")
    for r in rows:
        print(
            f"{r['file']:<32} {r['minutes']:7.2f}  {r['tag']:<10} {r['left_min']:9.2f} {r['right_min']:10.2f}"
        )
    print(
        f"-- 합계: left={totals['left']:.2f} min | right={totals['right']:.2f} min | total={totals['total']:.2f} min"
    )


# (선택) 판다스 표/CSV 저장
def to_dataframe(rows):
    return pd.DataFrame(
        rows, columns=["file", "minutes", "tag", "left_min", "right_min"]
    )


def save_summary_csv(prefix, rows, totals, out_dir="outputs"):
    os.makedirs(out_dir, exist_ok=True)
    df = to_dataframe(rows)
    df.to_csv(
        os.path.join(out_dir, f"{prefix}_per_file.csv"),
        index=False,
        encoding="utf-8-sig",
    )
    pd.DataFrame([totals]).to_csv(
        os.path.join(out_dir, f"{prefix}_totals.csv"), index=False, encoding="utf-8-sig"
    )


# =========================
# 테스터·모션별 분량 요약
# =========================
def summarize_by_tester_motion(df_list, paths, fs=FS):
    """
    df_list : load_group에서 나온 DataFrame 리스트
    paths   : 각 df에 대응되는 파일 경로 리스트 (learn_data_paths / val_data_paths)
    fs      : 샘플링 주파수(Hz)

    return : rows 리스트(dict) - [ {tester, motion, minutes}, ... ]
    """
    stats = defaultdict(float)

    for df, p in zip(df_list, paths):
        path = Path(p)
        parts = path.parts

        # 'tester1', 'tester2', ... 자동 추출
        tester = next((part for part in parts if part.startswith("tester")), "unknown")

        stem = path.stem.lower()

        # 모션 종류 추출 (outdoor 먼저 체크)
        if stem.startswith("out_looking"):
            motion = "out_looking"
        elif stem.startswith("out_swing"):
            motion = "out_swing"
        elif "looking" in stem:
            motion = "looking"
        elif "swing" in stem:
            motion = "swing"
        elif "calling" in stem:
            motion = "calling"
        else:
            motion = "unknown"

        minutes = len(df) / fs / 60.0
        stats[(tester, motion)] += minutes

    rows = []
    for (tester, motion), mins in sorted(stats.items()):
        rows.append(
            {
                "tester": tester,
                "motion": motion,
                "minutes": round(mins, 2),
            }
        )
    return rows


def pretty_print_tester_motion(rows, title="테스터·모션별 분량 요약"):
    print(f"\n=== {title} ===")
    print(f"{'tester':10} {'motion':15} {'minutes':>8}")
    for r in rows:
        print(f"{r['tester']:10} {r['motion']:15} {r['minutes']:8.2f}")


In [ ]:
# =========================
# 실행부
# =========================
if __name__ == "__main__":
    loader1 = lambda path, **opts: DataProcessor.load_and_preprocess_csv(path, **opts)

    # =========================
    # 학습 데이터 로딩 & 요약
    # =========================
    print("=== 학습 데이터 로딩 ===")
    learn_df_list, learn_X_list, learn_Y_list, learn_min = load_group(
        learn_data_paths,
        loader_fn=loader1,
        cfg=config,
        default_cfg=default_config,
    )
    print(f"[학습 전체 합계] {learn_min:.2f} 분")

    # 테스터·모션별 학습 분량 요약
    learn_rows_tm = summarize_by_tester_motion(learn_df_list, learn_data_paths, fs=FS)
    pretty_print_tester_motion(learn_rows_tm, title="학습 데이터 (테스터·모션별)")

    # 좌/우/혼합 기준 요약이 필요하면 그대로 사용 가능
    # learn_rows_turn, learn_totals_turn = summarize_turn_minutes(learn_df_list, learn_data_paths, fs=FS)
    # pretty_print_summary("학습 데이터 (좌/우/혼합 요약)", learn_rows_turn, learn_totals_turn)

    # =========================
    # 검증 데이터 로딩 & 요약
    # =========================
    print("\n=== 검증 데이터 로딩 ===")
    val_df_list, val_X_list, val_Y_list, val_min = load_group(
        val_data_paths,
        loader_fn=loader1,
        cfg=config,
        default_cfg=default_config,
    )
    print(f"[검증 전체 합계] {val_min:.2f} 분")

    # 테스터·모션별 검증 분량 요약
    val_rows_tm = summarize_by_tester_motion(val_df_list, val_data_paths, fs=FS)
    pretty_print_tester_motion(val_rows_tm, title="검증 데이터 (테스터·모션별)")

    # 검증 데이터 좌/우/혼합 요약도 필요하면 사용
    # val_rows_turn, val_totals_turn = summarize_turn_minutes(val_df_list, val_data_paths, fs=FS)
    # pretty_print_summary("검증 데이터 (좌/우/혼합 요약)", val_rows_turn, val_totals_turn)